In [1]:
import embedding
from transformers import BertTokenizer
from tqdm import tqdm

In [2]:
import pickle
import torch
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\picklefiles\\final_eal.pkl", 'rb') as eal:
    data = pickle.load(eal)

ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [3]:
ent[0]

{'id': '0',
 'target_entity': 'Gautama Buddha',
 'paragraph': "Srivastava's discovery of the terracotta sealings bearing the name Kapilavastu has led some scholars to believe that modern-day Piprahwa was the site of the ancient city of Kapilavastu, the capital of the Shakya kingdom, where Siddhartha Gautama spent the first 29 years of his life. Others suggest that the original site of Kapilavastu is located 16 km to the northwest, at Tilaurakot, in what is currently Kapilvastu District in Nepal. This question is especially important to scholars of Buddhist history, as Kapilavastu was the capital of the Shakya kingdom. King Śuddhodana and Queen Māyādevī lived at Kapilavastu, as did their son Prince Siddhartha Gautama until he left the palace at 29 years of age.",
 'entities': [{'eid': '00',
   'entity': 'Kapilavastu (ancient city)',
   'mention': 'Kapilavastu'},
  {'eid': '01', 'entity': 'Shakya', 'mention': 'Shakya'},
  {'eid': '02', 'entity': 'Gautama Buddha', 'mention': 'Siddhartha G

In [4]:
asp[0]

{'id': '0',
 'true_aspect': 'Biography',
 'candidate_aspects': [{'id': 'A00',
   'aspect_name': 'Historical Siddhārtha Gautama',
   'section_heading': ['Historical Siddhārtha Gautama'],
   'content': 'Scholars are hesitant to make unqualified claims about the historical facts of the Buddha\'s life. Most people accept that the Buddha lived, taught, and founded a monastic order during the Mahajanapada era during the reign of Bimbisara (, or c. 400 BCE), the ruler of the Magadha empire, and died during the early years of the reign of Ajatasatru, who was the successor of Bimbisara, thus making him a younger contemporary of Mahavira, the Jain tirthankara. While the general sequence of "birth, maturity, renunciation, search, awakening and liberation, teaching, death" is widely accepted, there is less consensus on the veracity of many details contained in traditional biographies.\nThe times of Gautama\'s birth and death are uncertain. Most historians in the early 20th century dated his lifeti

In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
target_emb = torch.zeros(len(ent), 100)
pretrained = 'bert-base-uncased'
ent_emb = embedding.EntityEmbedding(pretrained = pretrained)
for i in range(len(ent)):
    entity_word = ent[i]['target_entity']
    tokens = tokenizer.tokenize(entity_word)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0) 
    target_emb[i] = ent_emb(input_ids)
    
    

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
count = 0
for item in ent:
    count += len(item['entities'])
    
count

467

In [7]:
t_ent_emb = torch.zeros(count, 100)

for i in range(len(ent)):
    for el in ent[i]['entities']:
        word = el['entity']
        tokens = tokenizer.tokenize(word)
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        input_ids = torch.tensor(input_ids).unsqueeze(0)
        t_ent_emb[i] = ent_emb(input_ids, dim = 100)

In [8]:
taspc = 0
for el in asp:
    taspc += len(el['candidate_aspects']) 

In [9]:
taspc

639

In [10]:
asp_emb = torch.zeros(taspc, 100)
i = 0
j = 0
while(i < taspc):
    aspect = asp[j]['true_aspect']
    #print(j)
    tokens = tokenizer.tokenize(aspect)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0) 
    #print(i)
    asp_emb[i] = ent_emb(input_ids, dim = 100)
    #print(i, "first")
    i += 1
    for cand in asp[j]['candidate_aspects']:
        if i >= taspc:
            break
        if cand['aspect_name'] != aspect:
            casp = cand['aspect_name']
            #print(cand['aspect_name'], aspect)
            tokens = tokenizer.tokenize(casp)
            input_ids = tokenizer.convert_tokens_to_ids(tokens)
            input_ids = torch.tensor(input_ids).unsqueeze(0) 
            asp_emb[i] = ent_emb(input_ids, dim = 100)
            #print(i, "second")
            i+=1
    j += 1
        

In [12]:
asp_emb.shape

torch.Size([639, 100])

In [70]:
asp_count = 0
for item in asp:
    for el in item['candidate_aspects']:
        asp_count += len(el['entities'])
asp_count

18252

In [71]:
a_ent_emb = torch.zeros(asp_count, 100)
for i in tqdm(range(len(asp))):
    for el in asp[i]['candidate_aspects']:
        for ent in el['entities']:
            word = ent['entity_name']
            tokens = tokenizer.tokenize(word)
            input_ids = tokenizer.convert_tokens_to_ids(tokens)
            input_ids = torch.tensor(input_ids).unsqueeze(0) 
            a_ent_emb[i] = ent_emb(input_ids, dim = 100)
            

100%|██████████████████████████████████████████████████████████████████████████████| 100/100 [2:19:40<00:00, 83.81s/it]


In [15]:
import pickle
def dump(path, file, data):
    with open(f'{path}\\{file}.pkl', 'wb') as f:
        pickle.dump(data, f)
        print('Dumped successfully')

In [43]:
path = "picklefiles"


In [55]:
#dump(path, 'emb_aspectentities_random', asp_emb)
#dump(path, 'emb_targetentities_random', target_emb)
dump(path, 'emb_targetentities', target_emb)
dump(path, 'emb_aspectentities', asp_emb)

Dumped successfully
Dumped successfully


In [73]:
dump(path, 't_entities', t_ent_emb)
dump(path, 'a_entities', a_ent_emb)

Dumped successfully
Dumped successfully
Dumped successfully
Dumped successfully
